# Evaluate ComBat!
Can we freeze parameters of ComBat and achieve a similar normalization? What other approach can we take?

In [ ]:
import scanpy as sc
import sys
sys.path.append("../..")
from src.training import helpers as tr_h
from scgpt.preprocess import Preprocessor

# variables
data_path =  "/aloy/home/ddalton/projects/scGPT_playground/data/pp_data-25-08-14-01/data.h5ad"
gene_filtering = "top_presence"
scgpt_pp = "norm_log1p"

# load adata
adata = sc.read_h5ad(data_path)
print("adata loaded")
print(adata.X)


#! REMOVE - THIS IS A QUICK AND UGLY FIX
# only_control = True
# if only_control:
#     adata = adata[adata.obs["doid_id"] != "Control"].copy()

# define celltype as disease
adata.obs["celltype"] = adata.obs["do_id"].astype("category")

# generate celltype label
celltype_id_labels = adata.obs["celltype"].astype("category").cat.codes.values
celltypes = adata.obs["celltype"].unique()
num_types = len(np.unique(celltype_id_labels))
id2type = dict(enumerate(adata.obs["celltype"].astype("category").cat.categories))
adata.obs["celltype_id"] = celltype_id_labels




# gene filtering


if gene_filtering == "top_presence":
    mask_genes = tr_h.get_top_k_most_present_genes(
        adata, k=3501
    )
#! CLEAN UP
elif gene_filtering == "top_logfc":
    def compute_FC(adata)->np.array:
        # Subset by condition
        adata_control = adata[adata.obs["doid_id"] == "Control"]
        adata_disease = adata[adata.obs["doid_id"] != "Control"]

        # Mean expression per group (already log scale)
        mean_control = np.asarray(adata_control.X.mean(axis=0)).flatten()
        mean_disease = np.asarray(adata_disease.X.mean(axis=0)).flatten()

        # log2FC is simply difference of means
        log2fc = mean_disease - mean_control
        return log2fc


    def get_top_k_fc(adata, k=3500):


        total_n_genes = adata.X.shape[1]

        # compute fc
        all_fc = []
        for dsaid in adata.obs["dsaid"].unique():
            adata_dsaid = adata[adata.obs["dsaid"] == dsaid].copy()
            fc = compute_FC(adata_dsaid)
            all_fc.append(fc)

        all_fc = np.array(all_fc)

        # compute presence/absence of counts
        count_presence = np.sum(~np.isnan(adata.X), axis=0)


        # cutoff for top 50% genes by counts
        cutoff = np.percentile(count_presence, 50)
        mask_top = count_presence >= cutoff

        # compute max abs FC per gene
        max_abs_fc = np.nanmean(np.abs(all_fc), axis=0)

        # apply mask
        gene_idx_top = np.where(mask_top)[0]
        fc_sel = max_abs_fc[mask_top]

        # pick top K
        top_local = np.argsort(fc_sel)[::-1][:k]
        selected_idx = gene_idx_top[top_local]


        # boolean mask for the top `k` genes
        mask_top_k = np.zeros(total_n_genes, dtype=bool)
        mask_top_k[selected_idx] = True

        return mask_top_k

    mask_genes = get_top_k_fc(adata, k=3501)



logging.info(f"Combined mask {np.sum(mask_genes)} genes left")

# mask the genes
adata = adata[:, mask_genes]


# mask samples
# non_nan_percentage = np.sum(~np.isnan(adata.X), axis=1) / adata.X.shape[1]
# non_zero_non_nan_mask = ~np.isnan(adata.X) & ~(adata.X == 0)
non_zero_non_nan_mask = ~np.isnan(adata.X) 

non_zero_non_nan_mask_pct = np.sum(non_zero_non_nan_mask, axis=1) / adata.X.shape[1]

# mask samples that have less than 30% non-NaN values
mask_samples = non_zero_non_nan_mask_pct >= 0.3
logging.info(
    f"Filtering out {np.sum(~mask_samples)} / {len(mask_samples)} samples with less than 30% non-NaN values"
)

# apply the mask to the AnnData object
adata = adata[mask_samples, :]


# config parameters
data_is_raw = True
filter_gene_by_counts = False

if scgpt_pp == "norm_log1p":
    # set up the preprocessor, use the args to config the workflow
    preprocessor = Preprocessor(
        use_key="X",  # the key in adata.layers to use as raw data
        filter_gene_by_counts=False,  # step 1
        filter_cell_by_counts=False,  # step 2
        normalize_total=1e4,  # 3. whether to normalize the raw data and to what sum
        result_normed_key="X_normed",  # the key in adata.layers to store the normalized data
        log1p=True,  # 4. whether to log1p the normalized data
        result_log1p_key="X_log1p",
        subset_hvg=False,  # 5. whether to subset the raw data to highly variable genes
        hvg_flavor="seurat_v3" if True else "cell_ranger",
        binning=n_bins,  # 6. whether to bin the raw data and to what number of bins
        result_binned_key="X_binned",  # the key in adata.layers to store the binned data
        )
    
    # define mapping of input layer
    d_input_layer = {  # the values of this map coorespond to the keys in preprocessing
                    "normed_raw": "X_normed",
                    "log1p": "X_normed",
                    "binned": "X_binned",
                    }

elif scgpt_pp == "raw":

    # set up the preprocessor, use the args to config the workflow
    preprocessor = Preprocessor(
        use_key="X",  # the key in adata.layers to use as raw data
        filter_gene_by_counts=False,  # step 1
        filter_cell_by_counts=False,  # step 2
        normalize_total=None,  # 3. whether to normalize the raw data and to what sum
        result_normed_key="X_normed",  # the key in adata.layers to store the normalized data
        log1p=False,  # 4. whether to log1p the normalized data
        result_log1p_key="X_log1p",
        subset_hvg=False,  # 5. whether to subset the raw data to highly variable genes
        hvg_flavor="seurat_v3" if True else "cell_ranger",
        binning=51,  # 6. whether to bin the raw data and to what number of bins
        result_binned_key="X_binned",  # the key in adata.layers to store the binned data
        )
    
    # define mapping of input layer
    d_input_layer = {  # the values of this map coorespond to the keys in preprocessing
                    "normed_raw": "X",
                    "log1p": "X",
                    "binned": "X_binned"
                    }





# perform split
df_obs = adata.obs
new_obs = tr_h.split_stratified(
    df=df_obs,
    y_label="celltype",
    group_label="dataset_id",
    split_size=10,
)
tr_h.report_split(new_obs)
# update observations with new column split
adata.obs = new_obs


ImportError: cannot import name 'train_helper' from 'src.training' (unknown location)

In [ ]:
sys.path.append("../../")
from src.utils import utils as u


IndentationError: unexpected indent (3568762152.py, line 2)

In [ ]:
ls ../../src/training/